# Experiment 1: Manual classifier with decision boundaries (Iris)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Hand-written classifier: assign each point to the class with the nearest mean (centroid)
class NearestCentroid:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroids_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        return self

    def predict(self, X):
        dists = np.linalg.norm(X[:, None, :] - self.centroids_[None, :, :], axis=2)
        return self.classes_[np.argmin(dists, axis=1)]

iris = load_iris()
X, y = iris.data, iris.target

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = NearestCentroid().fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, average="macro"), 3))
print("Recall   :", round(recall_score(y_test, y_pred, average="macro"), 3))
print("F1-score :", round(f1_score(y_test, y_pred, average="macro"), 3))

In [ ]:
# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_acc = []
for fold, (tr, te) in enumerate(kf.split(X), 1):
    m = NearestCentroid().fit(X[tr], y[tr])
    acc = accuracy_score(y[te], m.predict(X[te]))
    fold_acc.append(acc)
    print(f"Fold {fold}: {acc:.3f}")
print(f"Mean accuracy: {np.mean(fold_acc):.3f} (+/- {np.std(fold_acc):.3f})")

In [ ]:
# Decision boundaries on two features so we can plot them
X2 = iris.data[:, :2]
m2 = NearestCentroid().fit(X2, y)

xx, yy = np.meshgrid(
    np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 300),
    np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 300),
)
Z = m2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
plt.scatter(X2[:, 0], X2[:, 1], c=y, edgecolor="k", cmap="viridis")
plt.xlabel("Sepal length")
plt.ylabel("Sepal width")
plt.title("Nearest-centroid decision boundaries (Iris)")
plt.show()